# 03 - Recovery Workflow Simulation

Simulate the full agent pipeline over a synthetic batch and measure:
- recovery rate by strategy
- cost per successful recovery
- ROI of the outreach mix

Requires the API running (`uvicorn main:app`) or uses the in-process supervisor.

In [ ]:
import asyncio, sys
sys.path.insert(0, '..')

from app.database.session import init_db, dispose_db, get_session_factory
from app.agents import get_supervisor
from app.schemas.payment_schemas import PaymentIngestRequest
from app.utils.mock_data import generate_payment_batch

init_db()

In [ ]:
async def simulate(n=200):
    db = get_session_factory()()
    supervisor = get_supervisor()
    stats = {'executed': 0, 'succeeded': 0, 'cost': 0, 'recovered': 0}
    try:
        for payload in generate_payment_batch(n, seed=11, success_ratio=0.0):
            req = PaymentIngestRequest(**payload)
            record, _ = await supervisor.ingest_payment(db, req)
            try:
                plan, rec = await supervisor.build_plan(db, record.id)
                if plan.strategy.value == 'write_off':
                    continue
                result, rec = await supervisor.execute_recovery(db, plan_id=rec.id)
                stats['executed'] += 1
                stats['succeeded'] += int(result.success)
                stats['cost'] += result.total_cost_paise
                stats['recovered'] += result.recovered_amount_paise
            except Exception as exc:
                print('skip', record.id, exc)
        db.commit()
    finally:
        db.close()
    return stats

stats = await simulate(150)
rate = stats['succeeded'] / max(stats['executed'], 1)
roi = (stats['recovered'] - stats['cost']) / max(stats['cost'], 1)
print(f"executed={stats['executed']} success_rate={rate:.1%} "
      f"cost=Rs {stats['cost']/100:.2f} recovered=Rs {stats['recovered']/100:.2f} ROI={roi:.1f}x")

**Interpretation**
- `smart_retry` plans are free (no channel cost) -> any conversion is pure upside.
- IVR steps cost ~Rs 12; they should only fire for high-value/high-risk segments (the strategist enforces this).
- Compare simulated ROI against your real `metrics/cost-analysis` after a week in production.

In [ ]:
dispose_db()